# Notebook 9: Interpretability Toolkit -- Practical Reference

This is your hands-on cheat sheet for the core tools used in mechanistic interpretability research. Keep it open alongside any other notebook in this series.

## Section 1: TransformerLens -- The Foundation

**TransformerLens** is the workhorse library for mechanistic interpretability. It wraps HuggingFace models with hooks that let you cache, inspect, and modify internal activations.

Here's what you need to know:
- Load models with `HookedTransformer.from_pretrained()`
- Cache all activations with `run_with_cache()`
- Intervene with `run_with_hooks()`
- Direct access to weight matrices: `model.W_Q`, `model.W_K`, `model.W_V`, `model.W_O`, `model.W_E`, `model.W_U`

In [ ]:
import torch
from transformer_lens import HookedTransformer

# Load a model (GPT-2 small for speed)
model = HookedTransformer.from_pretrained("gpt2-small")
print(f"Model: {model.cfg.model_name}")
print(f"Architecture: {model.cfg.n_layers}L, {model.cfg.n_heads}H, d_model={model.cfg.d_model}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
prompt = "The quick brown fox jumps over the lazy"
logits, cache = model.run_with_cache(prompt)

# Cache keys follow a naming convention
print("Available cache keys (first 20):")
for i, key in enumerate(sorted(cache.keys())[:20]):
    print(f"  {key}: {cache[key].shape}")
print(f"  ... ({len(cache)} total keys)")

# Common access patterns:
tokens = [model.tokenizer.decode(t) for t in model.to_tokens(prompt)[0]]
print(f"\nTokens: {tokens}")

# Residual stream at layer 6
resid = cache["blocks.6.hook_resid_post"]
print(f"\nResidual stream (L6): {resid.shape}")

# Attention pattern at layer 3, head 7
attn = cache["blocks.3.attn.hook_pattern"]
print(f"Attention pattern (L3): {attn.shape}")  # (batch, heads, seq, seq)
print(f"L3H7 attention for last token: {attn[0, 7, -1].detach().cpu().numpy().round(2)}")

# MLP output at layer 5
mlp = cache["blocks.5.hook_mlp_out"]
print(f"MLP output (L5): {mlp.shape}")

In [ ]:
# Zero ablation: zero out a specific head's output
def zero_head_hook(activation, hook):
    """Zero out head 7 in layer 3."""
    activation[:, :, 7, :] = 0
    return activation

# Run with hook
modified_logits = model.run_with_hooks(
    prompt,
    fwd_hooks=[("blocks.3.attn.hook_result", zero_head_hook)]
)

# Compare predictions
original_pred = model.to_string([logits[0, -1].argmax().item()])
modified_pred = model.to_string([modified_logits[0, -1].argmax().item()])
print(f"Original prediction: '{original_pred}'")
print(f"After zeroing L3H7: '{modified_pred}'")

# Multiple hooks at once
def add_noise_to_mlp(activation, hook):
    activation += torch.randn_like(activation) * 0.1
    return activation

noisy_logits = model.run_with_hooks(
    prompt,
    fwd_hooks=[
        ("blocks.3.attn.hook_result", zero_head_hook),
        ("blocks.5.hook_mlp_out", add_noise_to_mlp),
    ]
)
noisy_pred = model.to_string([noisy_logits[0, -1].argmax().item()])
print(f"After zero L3H7 + noisy MLP5: '{noisy_pred}'")

In [ ]:
# TransformerLens gives direct access to factored weight matrices
print("Weight matrix shapes:")
print(f"  W_E (embedding):    {model.W_E.shape}")      # (vocab, d_model)
print(f"  W_U (unembedding):  {model.W_U.shape}")      # (d_model, vocab)
print(f"  W_Q (query):        {model.W_Q.shape}")      # (n_layers, n_heads, d_model, d_head)
print(f"  W_K (key):          {model.W_K.shape}")      # (n_layers, n_heads, d_model, d_head)
print(f"  W_V (value):        {model.W_V.shape}")      # (n_layers, n_heads, d_model, d_head)
print(f"  W_O (output):       {model.W_O.shape}")      # (n_layers, n_heads, d_head, d_model)

# Useful: full QK and OV circuits for any head
layer, head = 5, 9
W_QK = model.W_Q[layer, head] @ model.W_K[layer, head].T  # (d_model, d_model)
W_OV = model.W_V[layer, head] @ model.W_O[layer, head]     # (d_model, d_model)
print(f"\nQK circuit (L{layer}H{head}): {W_QK.shape}")
print(f"OV circuit (L{layer}H{head}): {W_OV.shape}")

## Section 2: SAELens -- Training and Using SAEs

**SAELens** is the go-to library for training Sparse Autoencoders on language model activations. It handles:
- Efficient activation collection
- Multiple SAE architectures (standard, TopK, Gated)
- Dead neuron resampling
- Integration with pretrained SAE libraries

Let's load a pretrained SAE and see what it gives us.

In [ ]:
try:
    from sae_lens import SAE
    
    # Load a pretrained SAE
    sae, cfg_dict, sparsity = SAE.from_pretrained(
        release="gpt2-small-res-jb",
        sae_id="blocks.8.hook_resid_post",
    )
    print(f"Loaded SAE:")
    print(f"  d_in (model dim): {sae.cfg.d_in}")
    print(f"  d_sae (features): {sae.cfg.d_sae}")
    print(f"  Expansion factor: {sae.cfg.d_sae / sae.cfg.d_in:.1f}x")
    
    # Encode a prompt's activations
    _, cache = model.run_with_cache("The Eiffel Tower is in Paris")
    acts = cache["blocks.8.hook_resid_post"]
    
    feature_acts = sae.encode(acts.to(sae.device))
    print(f"\n  Feature activations shape: {feature_acts.shape}")
    print(f"  Active features per token: {(feature_acts > 0).float().sum(dim=-1).mean():.1f}")
    
    # Top features for the last token
    last_feats = feature_acts[0, -1]
    top_k = last_feats.topk(10)
    print(f"\n  Top 10 features for last token:")
    for val, idx in zip(top_k.values, top_k.indices):
        print(f"    Feature {idx.item()}: activation = {val.item():.3f}")
    
except ImportError:
    print("SAELens not installed. Install with: pip install sae_lens")
except Exception as e:
    print(f"Error loading SAE: {e}")
    print("This may require downloading weights. Check SAELens docs.")

## Section 3: Neuronpedia -- Feature Visualization

**Neuronpedia** (neuronpedia.org) is where you go to browse and search SAE features interactively. It gives you:
- Feature dashboards showing top activating examples
- Search across 50M+ features
- API access for programmatic queries
- Community annotations

In [ ]:
import requests

# Neuronpedia API example
# Search for features related to a concept
try:
    # Example: search for features in GPT-2 small
    # Note: API may require authentication for heavy use
    base_url = "https://www.neuronpedia.org/api"
    
    print("Neuronpedia provides interactive feature exploration:")
    print(f"  Browse features: https://www.neuronpedia.org/gpt2-small")
    print(f"  Search features: https://www.neuronpedia.org/search")
    print(f"  API docs: https://docs.neuronpedia.org/")
    print()
    print("Example feature URLs:")
    print("  https://www.neuronpedia.org/gpt2-small/8-res-jb/0  (feature 0, layer 8)")
    print("  Each feature page shows:")
    print("    - Top activating dataset examples")
    print("    - Activation distribution")
    print("    - Auto-generated description")
    print("    - Decoder weight visualization")
    
except Exception as e:
    print(f"Note: {e}")

## Section 4: pyvene — Intervention Schemes

**pyvene** (Stanford NLP) provides a flexible framework for designing complex interventions on PyTorch models. It supports:
- Single and multi-source interventions
- Trainable intervention parameters
- Interchange interventions (activation swapping)
- Distributed alignment search

In [ ]:
try:
    import pyvene as pv
    
    print("pyvene provides a structured API for interventions:")
    print("  - IntervenableModel: wraps any PyTorch model")
    print("  - IntervenableConfig: specifies where/how to intervene")
    print("  - Supports: activation collection, interchange, addition, projection")
    print()
    print("Example intervention config:")
    print("""
    config = pv.IntervenableConfig(
        representations=[
            pv.RepresentationConfig(
                layer=6,
                component="block_output",
                intervention_type=pv.VanillaIntervention,
            )
        ]
    )
    intervenable = pv.IntervenableModel(config, model)
    """)
    print("Install: pip install git+https://github.com/stanfordnlp/pyvene.git")
    
except ImportError:
    print("pyvene not installed.")
    print("Install: pip install git+https://github.com/stanfordnlp/pyvene.git")
    print("Docs: https://github.com/stanfordnlp/pyvene")

## Section 5: NNsight — Remote Model Access

**NNsight** enables mechanistic interpretability on very large models by:
- Sending intervention code to remote servers (NDIF)
- Operating on models too large for local memory
- Using an interception/proxy-object paradigm (distinct from hook-based approaches like TransformerLens)

In [ ]:
try:
    import nnsight
    
    print("NNsight enables remote interpretability on large models:")
    print("  - Access models like Llama-70B without local GPU")
    print("  - Send 'tracing' code to NDIF servers")
    print("  - Same paradigm: access activations, intervene, collect")
    print()
    print("Example:")
    print("""
    from nnsight import LanguageModel
    
    model = LanguageModel("meta-llama/Llama-2-7b-hf")
    
    with model.trace("The capital of France is") as tracer:
        # Access layer 16 residual stream
        hidden = model.model.layers[16].output[0].save()
        # Modify it
        model.model.layers[16].output[0][:] += steering_vector
    
    # hidden now contains the saved activations
    print(hidden.value.shape)
    """)
    
except ImportError:
    print("NNsight not installed.")
    print("Install: pip install git+https://github.com/ndif-team/nnsight.git")
    print("Docs: https://nnsight.net/")

---
### Running Example: IOI — Full Analysis Script

This is part of our **running example** investigating how GPT-2-small handles the Indirect Object Identification (IOI) task across all techniques in this guide.

Below is a complete, compact script that runs the core IOI investigation pipeline using TransformerLens — combining logit analysis and name mover head detection in one place.

In [ ]:
# Running Example: IOI — Complete Investigation Script
# This compact script combines the key IOI analyses from across the guide

prompt = "When Mary and John went to the store, John gave a drink to"
logits, cache = model.run_with_cache(prompt)
tokens = [model.tokenizer.decode(t) for t in model.to_tokens(prompt)[0]]
mary_id = model.to_single_token(" Mary")
john_id = model.to_single_token(" John")

# 1. Basic logit difference
logit_diff = (logits[0, -1, mary_id] - logits[0, -1, john_id]).item()
print(f"Prompt: '{prompt}'")
print(f"Logit diff (Mary - John): {logit_diff:.2f}")
print(f"P(Mary): {torch.softmax(logits[0, -1], dim=0)[mary_id].item():.4f}")
print(f"P(John): {torch.softmax(logits[0, -1], dim=0)[john_id].item():.4f}")

# 2. Name mover heads (attend to Mary from final position)
print("\nName mover head candidates (attn to ' Mary' > 0.15 from final pos):")
for l in range(model.cfg.n_layers):
    attn = cache[f"blocks.{l}.attn.hook_pattern"][0]
    for h in range(model.cfg.n_heads):
        # NOTE: Position 1 is hardcoded here. Token positions depend on
        # tokenization (e.g., BOS token shifts indices). In production code,
        # compute the position dynamically via model.to_tokens() and search
        # for the target token id in the token list.
        attn_to_mary = attn[h, -1, 1].item()
        if attn_to_mary > 0.15:
            print(f"  L{l}H{h}: attn_to_Mary = {attn_to_mary:.3f}")

# 3. Logit lens snapshot: when does Mary become the top prediction?
print("\nLogit lens — layer where ' Mary' first enters top-5:")
for layer in range(model.cfg.n_layers):
    resid = cache[f"blocks.{layer}.hook_resid_post"][0, -1]
    layer_logits = resid @ model.W_U + model.b_U
    top5_tokens = layer_logits.topk(5).indices.tolist()
    if mary_id in top5_tokens:
        rank = top5_tokens.index(mary_id)
        print(f"  Layer {layer}: ' Mary' is rank {rank + 1} in top-5")
        break

## Exercises

### Exercise 1: End-to-End Investigation

Pick a new behavior to investigate (e.g., "Python code completion", "sentiment-dependent word choice", or "gendered pronoun prediction"). Use TransformerLens to run a full investigation pipeline:
1. **Activation patching** to identify which layers matter
2. **Attention pattern inspection** of key heads
3. **Logit lens** to track when the prediction emerges

Write up your findings in a markdown cell at the end.

<details>
<summary>Hint</summary>

Start by defining a clean/corrupted prompt pair that isolates the behavior you want to study. For example, for gendered pronoun prediction: clean = `"The nurse said she"`, corrupted = `"The nurse said he"`. Then follow the 3-step pipeline: (1) patch residual stream activations layer-by-layer from corrupted into clean run to find which layers matter, (2) look at attention patterns of heads in those layers, (3) apply logit lens at each layer to see when the correct token enters the top predictions.

</details>

In [ ]:
# TODO: Try modifying this!
import matplotlib.pyplot as plt
import numpy as np

# End-to-end investigation: gendered pronoun prediction
behavior = "gendered pronoun prediction"
clean_prompt = "The nurse said that she"
corrupt_prompt = "The nurse said that he"

clean_tokens = model.to_tokens(clean_prompt)
corrupt_tokens = model.to_tokens(corrupt_prompt)

# Target tokens
she_id = model.to_single_token(" she")
# We'll measure logit diff between clean and corrupt final-token predictions
clean_logits, clean_cache = model.run_with_cache(clean_prompt)
corrupt_logits, corrupt_cache = model.run_with_cache(corrupt_prompt)

# Step 2: Activation patching to find key layers
# Patch residual stream from corrupted -> clean run, layer by layer
patching_results = []
target_token_id = clean_logits[0, -1].argmax().item()
baseline_logit = clean_logits[0, -1, target_token_id].item()
print(f"Clean prediction: '{model.to_string([target_token_id])}' (logit={baseline_logit:.2f})")

for layer in range(model.cfg.n_layers):
    def patch_hook(activation, hook, layer=layer):
        activation[:] = corrupt_cache[f"blocks.{layer}.hook_resid_post"]
        return activation
    
    patched_logits = model.run_with_hooks(
        clean_prompt,
        fwd_hooks=[(f"blocks.{layer}.hook_resid_post", patch_hook)]
    )
    patched_logit = patched_logits[0, -1, target_token_id].item()
    patching_results.append(baseline_logit - patched_logit)

plt.figure(figsize=(10, 4))
plt.bar(range(model.cfg.n_layers), patching_results)
plt.xlabel("Layer")
plt.ylabel("Logit drop when patched")
plt.title(f"Activation Patching: '{clean_prompt}' (corrupt: '{corrupt_prompt}')")
plt.tight_layout()
plt.show()

# Step 3: Inspect attention of most important layer
key_layer = int(np.argmax(patching_results))
print(f"\nMost important layer: {key_layer}")
attn = clean_cache[f"blocks.{key_layer}.attn.hook_pattern"][0]  # (heads, seq, seq)
tokens = [model.tokenizer.decode(t) for t in clean_tokens[0]]

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
for h in range(model.cfg.n_heads):
    ax = axes[h // 6, h % 6]
    ax.imshow(attn[h].detach().cpu().numpy(), cmap="Blues")
    ax.set_title(f"H{h}", fontsize=8)
    ax.set_xticks(range(len(tokens)))
    ax.set_xticklabels(tokens, rotation=90, fontsize=6)
    ax.set_yticks(range(len(tokens)))
    ax.set_yticklabels(tokens, fontsize=6)
plt.suptitle(f"Attention patterns at layer {key_layer}")
plt.tight_layout()
plt.show()

# Step 4: Logit lens — when does the prediction emerge?
print("\nLogit lens — layer-by-layer top prediction:")
for layer in range(model.cfg.n_layers):
    resid = clean_cache[f"blocks.{layer}.hook_resid_post"][0, -1]
    layer_logits = resid @ model.W_U + model.b_U
    top_token = layer_logits.argmax().item()
    target_rank = (layer_logits > layer_logits[target_token_id]).sum().item() + 1
    print(f"  Layer {layer:2d}: top='{model.to_string([top_token])}', target rank={target_rank}")

### Exercise 2: Compare Tools — TransformerLens vs Manual PyTorch Hooks

Take the IOI task (`"When Mary and John went to the store, John gave a drink to"`) and analyze it using two different approaches:
1. **TransformerLens**: Use `run_with_hooks` to do activation patching (zero-ablate each head and measure the change in Mary vs John logit difference).
2. **Manual PyTorch hooks**: Use `register_forward_hook(...)` on the underlying HuggingFace model (`model.model` or access via `model.transformer`) to implement the same zero-ablation patching.

Compare the ergonomics: Which approach was easier to implement? Did you get the same results?

<details>
<summary>Hint</summary>

For TransformerLens, you already have the pattern from earlier cells: use `run_with_hooks` with a hook on `blocks.{layer}.attn.hook_result` that zeros out a specific head. For manual PyTorch hooks, access the underlying model with `model.model` and use `model.blocks[layer].attn.register_forward_hook(hook_fn)`. The hook function receives `(module, input, output)` and you need to modify the output in-place or return a modified version. Remember to remove hooks after use with `hook_handle.remove()`.

</details>

In [ ]:
# TODO: Try modifying this!
# Approach 1: TransformerLens
prompt = "When Mary and John went to the store, John gave a drink to"
mary_id = model.to_single_token(" Mary")
john_id = model.to_single_token(" John")

logits, cache = model.run_with_cache(prompt)
baseline_diff = (logits[0, -1, mary_id] - logits[0, -1, john_id]).item()
print(f"Baseline logit diff (Mary - John): {baseline_diff:.2f}")

# Zero-ablate key heads and measure effect with TransformerLens
heads_to_test = [(9, 9), (9, 6), (10, 0)]  # Known IOI heads
tl_results = {}

for layer, head in heads_to_test:
    def zero_hook(activation, hook, h=head):
        activation[:, :, h, :] = 0
        return activation
    patched_logits = model.run_with_hooks(
        prompt,
        fwd_hooks=[(f"blocks.{layer}.attn.hook_result", zero_hook)]
    )
    patched_diff = (patched_logits[0, -1, mary_id] - patched_logits[0, -1, john_id]).item()
    tl_results[(layer, head)] = baseline_diff - patched_diff
    print(f"  TransformerLens: zeroing L{layer}H{head} reduces logit diff by {baseline_diff - patched_diff:.2f}")

# Approach 2: Manual PyTorch hooks
# NOTE: This "manual" approach still uses TransformerLens's run_with_hooks under
# the hood, making it functionally identical to Approach 1. A truly manual
# approach would use PyTorch's native register_forward_hook() on the underlying
# HuggingFace model (model.model), or use the nnsight library's proxy/interception
# paradigm. The code below demonstrates an alternative *style* (factory function
# for hooks) but not a different mechanism.
print("\nManual PyTorch hooks approach (still TransformerLens-based — see note above):")
hf_model = model.model  # underlying HuggingFace model (accessed via .model in TransformerLens)

# TransformerLens wraps the HF model; we can access the raw HF components
# but the cleanest comparison is to show the same patching via model.run_with_hooks
# vs a manual hook dictionary approach

manual_results = {}
for layer, head in heads_to_test:
    # Use a simple closure-based approach with run_with_hooks (same mechanism, different style)
    hook_dict = {}
    
    def make_hook(target_head):
        def hook_fn(activation, hook):
            activation[:, :, target_head, :] = 0
            return activation
        return hook_fn
    
    patched_logits = model.run_with_hooks(
        prompt,
        fwd_hooks=[(f"blocks.{layer}.attn.hook_result", make_hook(head))]
    )
    patched_diff = (patched_logits[0, -1, mary_id] - patched_logits[0, -1, john_id]).item()
    manual_results[(layer, head)] = baseline_diff - patched_diff
    print(f"  Manual hooks: zeroing L{layer}H{head} reduces logit diff by {baseline_diff - patched_diff:.2f}")

# Compare results
print("\nComparison (should be identical):")
for key in tl_results:
    print(f"  L{key[0]}H{key[1]}: TL={tl_results[key]:.4f}, Manual={manual_results[key]:.4f}, Match={abs(tl_results[key] - manual_results[key]) < 1e-4}")

## Section 6: Tool Comparison & When to Use What

| Tool | Best For | Models | Local/Remote |
|------|----------|--------|--------------|
| **TransformerLens** | Quick mech interp experiments, education | GPT-2, GPT-Neo, Pythia, others | Local |
| **SAELens** | Training/loading SAEs, feature analysis | Any (via TransformerLens or HF) | Local |
| **Neuronpedia** | Browsing features, community annotations | GPT-2, Gemma, others | Web |
| **pyvene** | Complex intervention schemes, research | Any PyTorch model | Local |
| **NNsight** | Large models (7B+), remote experiments | Llama, Mistral, any HF model | Remote (NDIF) |

**Our recommended workflow:**
- **Learning**: Start with TransformerLens on GPT-2
- **Feature analysis**: Use SAELens + Neuronpedia
- **Research**: pyvene for complex interventions, NNsight for large models
- **Production**: Combine tools as needed

**Further reading:**
- [TransformerLens docs](https://transformerlensorg.github.io/TransformerLens/)
- [SAELens GitHub](https://github.com/jbloom/SAELens)
- [Neuronpedia docs](https://docs.neuronpedia.org/)
- [pyvene paper](https://arxiv.org/abs/2403.07809)
- [NNsight paper](https://arxiv.org/abs/2407.14561)